In [1]:
#Start with notebook structure andinstall packages
#langchain helps connect PDFs, embeddings, vector databases, chat models
#langchain-open ai connects langchain to openai models
#langchain-communtiy gives doc laoders like PyPDFLoader
#langchain-chroma lets us use chromadb as our vector database
#chromadb is a vector database that stores embeddings and allows for fast similarity search
#pypdf helps read PDF Files
#gradio is a library for building web interfaces for machine learning models
#python-dotenv helps load environment variables from a .env file
!pip install langchain langchain-openai langchain-community langchain-chroma chromadb pypdf gradio python-dotenv

  Using cached langchain_community-0.4.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached langchain_chroma-1.1.0-py3-none-any.whl.metadata (1.9 kB)
  Using cached chromadb-1.5.9-cp39-abi3-macosx_11_0_arm64.whl.metadata (5.0 kB)
  Using cached langgraph_prebuilt-1.1.0-py3-none-any.whl.metadata (5.2 kB)
  Using cached ormsgpack-1.12.2-cp312-cp312-macosx_10_12_x86_64.macosx_11_0_arm64.macosx_10_12_universal2.whl.metadata (3.2 kB)
  Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl.metadata (14 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached httpx_sse-0.4.3-py3-none-any.whl.metadata (9.7 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached frozenlist-1.8.0-cp312-cp312-macosx_11_0_arm64.whl.metadata (20 kB)
  Using cached pypika-0.51.1-py2.py3-none-any.whl.metadata (51 kB)
  Using cached overrides-7.7.0-py3-none-any.whl.metadata (5.8 kB)
  Using cached importlib_resources-7.1.0-py3-none-any.whl.metadata (4.0 kB)
  U

In [2]:
#import libraries
import os #lets Python read environment variables
from dotenv import load_dotenv #loads API key from a .env file

from langchain_community.document_loaders import PyPDFLoader #PyPDFLoader loads the Nestle HR PDF
from langchain_text_splitters import RecursiveCharacterTextSplitter #Breaks the PDF text into smaller chunks

from langchain_openai import OpenAIEmbeddings, ChatOpenAI #OpenAIEmbeddings turns chunks to numbers ChatOpenAI connects to the GPT model
from langchain_chroma import Chroma #supported as LangChain vector store-stores and searches the text vectors

from langchain_classic.chains import RetrievalQA #RetrievalQA builds the question-answering chain
from langchain_core.prompts import PromptTemplate #PromptTemplate controls how the chatbot answers.

import gradio as gr #gradio builds the user interface.


/var/folders/n4/f1p3pr8s19d288h3t3_zy_nh0000gn/T/ipykernel_64273/3549342339.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader #PyPDFLoader loads the Nestle HR PDF
/opt/miniconda3/envs/myenv2/lib/python3.12/site-packages/requests/__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [3]:
import langchain

print(langchain.__version__)

1.3.4


In [4]:
!pip install langchain-classic

In [5]:
#Load OpenAI API key
load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

if api_key is None:
    print("OpenAI API key not found.")
else:
    print("OpenAI API key loaded successfully.")

OpenAI API key loaded successfully.


In [6]:
import os

print(os.getcwd())

/Users/maribethmiller/Downloads/Dataset 2


In [7]:
print(os.listdir())

['F6041A83-A302-4A93-8A63-AF09278E3EC6.olk15Message', 'amazon.csv', 'Final Capstone2.ipynb', 'nestle_hr_chroma', 'the_nestle_hr_policy_pdf_2012.pdf', 'chatbotiworkingimage.jpg', 'README.md', 'bikeease_ad_image.png', 'Unit6Project2.ipynb', 'Unit6IncrementalCapstone.ipynb', '.env', 'Unit6Project1.ipynb', 'Unit6NetflixImageGenerator.ipynb', 'Gradio_Documentation.pdf']


In [8]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

if api_key:
    print("API key loaded successfully!")
else:
    print("API key not found.")

API key loaded successfully!


In [9]:
import os

print(".env" in os.listdir())

True


In [10]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

api_key = os.getenv("OPENAI_API_KEY")

if api_key:
    print("API key loaded successfully!")
else:
    print("API key not found.")

API key loaded successfully!


In [11]:
import os

pdf_files = [file for file in os.listdir() if file.endswith(".pdf")]

print(pdf_files)

['the_nestle_hr_policy_pdf_2012.pdf', 'Gradio_Documentation.pdf']


In [12]:
from langchain_community.document_loaders import PyPDFLoader

In [13]:
#Load the PDF
pdf_path = "the_nestle_hr_policy_pdf_2012.pdf"

loader = PyPDFLoader(pdf_path)

documents = loader.load()

print("Number of pages loaded:", len(documents))
print(documents[0].page_content[:500])

Number of pages loaded: 8
Policy
Mandatory
September  2012
The Nestlé  
Human Resources Policy


In [14]:
#Split the PDF into chunks
#RecursiveCharacterTextSplitter breaks chunks into small pieces
#chunk is about 1000 pieces and nearby chunks share 150 characters so it doesn't lose context.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)
#apply splitter to the 8 pDF pages we loaded
chunks = text_splitter.split_documents(documents)

print("Number of chunks created:", len(chunks))
print(chunks[0].page_content[:500])#preview the first 500 characters of the first chunk

Number of chunks created: 20
Policy
Mandatory
September  2012
The Nestlé  
Human Resources Policy


There is now 8 PDF pages and 20 searchable chunks

In [15]:
#create embeddings and store them in chroma
embedding_model = OpenAIEmbeddings() #creates the model that turns text into numbers
#Chroma.from_documents() stores your 20 chunks in a searchable database.
#documents=chunks tells Chroma what text to store.
#embedding=embedding_model tells Chroma how to convert the text into vectors.
#persist_directory="./nestle_hr_chroma" saves the database in a folder.

vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="./nestle_hr_chroma"
)

print("Vector database created successfully!")

Vector database created successfully!


In [16]:
#Create the retriever
#turn  Chroma database into a search tool that finds the 3 most relevant chunks for a question.
retriever = vector_db.as_retriever(
    search_kwargs={"k": 3}
)

print("Retriever created successfully!")

Retriever created successfully!


In [17]:
#test it
test_question = "What is the Nestlé human resources policy about?"

results = retriever.invoke(test_question)

print("Number of chunks retrieved:", len(results))
print(results[0].page_content[:500])

Number of chunks retrieved: 3
Policy
Mandatory
September  2012
The Nestlé  
Human Resources Policy


Retriever is working

In [18]:
#Create the prompt template
prompt_template = """
You are an HR assistant for Nestlé.

Use only the information from the HR policy document to answer the question.
If the answer is not found in the document, say:
"I could not find that information in the provided HR policy document."

Context:
{context}

Question:
{question}

Answer:
"""

prompt = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

print("Prompt template created successfully!")

Prompt template created successfully!


In [19]:
#Build the question Answering Chain
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff",
    chain_type_kwargs={"prompt": prompt}
)

print("QA chain created successfully!")

QA chain created successfully!


In [20]:
#Chatbot brain is built - Now test it
question = "What is the Nestlé Human Resources Policy about?"

response = qa_chain.invoke({"query": question})

print(response["result"])

The Nestlé Human Resources Policy outlines the guidelines for effective Human Resources Management throughout the Nestlé Group globally. It emphasizes the importance of employee engagement as key to the company's success and explains the vision and mission of the Human Resources function, as well as every aspect of the employee lifecycle. The policy is built on the Nestlé Management and Leadership Principles and the Corporate Business Principles, which serve as foundational pillars. The implementation of the policy is guided by sound judgment, compliance with local market laws, and common sense.


In [21]:
#Next step is Gradio-The user-friendly chatbot interface
def hr_chatbot(message, history):
    response = qa_chain.invoke({"query": message})
    return response["result"]

demo = gr.ChatInterface(
    fn=hr_chatbot,
    title="Nestlé HR Policy Assistant",
    description="Ask questions about the Nestlé Human Resources Policy document."
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Load PDF

The Nestlé Human Resources Policy PDF was loaded using PyPDFLoader. This allowed the chatbot to access the document content for retrieval and question answering.

Text Chunking

The document was split into 20 chunks using RecursiveCharacterTextSplitter. Chunking improves retrieval accuracy by allowing the system to search smaller sections of the document rather than entire pages.

Embeddings and Chroma

OpenAI embeddings converted the text chunks into numerical vector representations. Chroma stored these vectors and enabled semantic search based on meaning rather than exact keyword matching.

RetrievalQA

The RetrievalQA chain combines document retrieval with a large language model. When a user submits a question, the system first retrieves relevant document chunks and then generates a response based on that information.

Gradio Interface

Gradio provides a user-friendly interface that allows users to interact with the chatbot and retrieve HR policy information through natural language questions.

ChatBot Tested

The ChatBot was tested to demonstate that it worked. Also tested was asking a question that was not in the HR Policy (What is Nestlé's stock price today?) to which the chatbot answered, "I could not find that information in the provided HR policy document." This demonstrates that the chatbot is using the PDF as its knowledge source rather than making up answers.